# Giai đoạn 1: Data Collection (Cào dữ liệu TMDB)

Notebook này **sinh hai file CSV** ngay trong thư mục `Notebook_Report/`:

- `cinesense_movies.csv`
- `cinesense_reviews.csv`

**Cách làm** bám **đúng code CineSense** trong `etl_pipeline/crawler.py`:

1. `TMDBClient.get_genres()` → map `genre_id` → tên thể loại.
2. Khám phá phim: `/movie/popular` hoặc `/movie/top_rated` (theo `MOVIE_SOURCE`), phân trang 20 phim/trang.
3. Với mỗi phim: `get_movie_reviews(movie_id, max_pages=…)` giống crawler — `max_pages` tính như `fetch_movies_with_reviews` trong cùng file (`(max_reviews_per_movie // 20) + 1`), rồi lấy tối đa `MAX_REVIEWS_PER_MOVIE` review.
4. Lọc review **cùng heuristic** với pipeline ETL trong `etl_pipeline/main.py` + `embedder.py`: bỏ rỗng, `is_noisy_review`, và (tuỳ chọn) chỉ giữ tiếng Anh / unknown như khi export `--only-english`.

**Schema CSV** (cột, thứ tự) khớp notebook 02 và phần seed/ETL CineSense — một định dạng cho toàn bộ báo cáo.

## Tham số cần chỉnh

- `MAX_MOVIES`: số phim tối đa (mặc định ~4900).
- `MOVIE_SOURCE`: `"popular"` hoặc `"top_rated"`.
- `MAX_REVIEWS_PER_MOVIE`: giới hạn review/phim (mặc định 2 để xấp xỉ ~9k review khi ~4.9k phim).
- `ONLY_ENGLISH_REVIEWS`: `True` để chỉ giữ review được heuristic xếp là tiếng Anh / unknown (giống ETL).

Cần **`TMDB_API_KEY`** trong `.env` ở **root repo** (hoặc biến môi trường) — giống `etl_pipeline`.

> Chạy notebook với **working directory là `Notebook_Report/`** (hoặc chỉnh `OUT_*` nếu cần).


In [ ]:
"""Cào TMDB bằng TMDBClient (etl_pipeline) và ghi 2 CSV đúng schema báo cáo."""
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "Notebook_Report" else _here
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv:
    load_dotenv(REPO_ROOT / ".env")

from etl_pipeline.crawler import TMDBClient, TMDBMovie, TMDBReview
from etl_pipeline.embedder import is_noisy_review
from etl_pipeline.main import _infer_review_language

# ---------------------------------------------------------------------------
# Tham số (cùng ý tưởng với etl_pipeline/crawler + main.run_tmdb_etl_pipeline)
# ---------------------------------------------------------------------------
MAX_MOVIES = 4900
MOVIE_SOURCE = "popular"  # "popular" | "top_rated"
MAX_REVIEWS_PER_MOVIE = 2
ONLY_ENGLISH_REVIEWS = True

OUT_MOVIES = "cinesense_movies.csv"
OUT_REVIEWS = "cinesense_reviews.csv"

MOVIE_PAGE_SIZE = 20
PAGES_END = math.ceil(MAX_MOVIES / MOVIE_PAGE_SIZE)


def movie_to_report_row(m: TMDBMovie, client: TMDBClient) -> dict:
    """Cột khớp scripts/export_dataset_csv_from_db.py (movies): sort genre names như export DB."""
    names = [client.get_genre_name(gid) for gid in (m.genre_ids or [])]
    genres_str = ", ".join(sorted({n for n in names if n}))
    return {
        "tmdb_id": m.tmdb_id,
        "title": m.title or "",
        "overview": m.overview or "",
        "genres": genres_str,
        "release_date": m.release_date.isoformat() if m.release_date else "",
        "poster_path": m.poster_path or "",
        "vote_average": m.vote_average if m.vote_average is not None else "",
        "vote_count": m.vote_count if m.vote_count is not None else "",
        "popularity": m.popularity if m.popularity is not None else "",
    }


def review_to_report_row(r: TMDBReview, movie_tmdb_id: int) -> dict:
    """Cột reviews; review id từ API nằm trong field TMDBReview.tmdb_id (DTO)."""
    return {
        "review_id": r.tmdb_id,
        "tmdb_id": movie_tmdb_id,
        "author": r.author or "",
        "author_name": r.author_name or "",
        "content": r.content or "",
        "rating": r.rating if r.rating is not None else "",
        "avatar_path": r.avatar_path or "",
        "created_at": r.created_at or "",
        "url": r.url or "",
        "source": "tmdb",
    }


movies_rows: list[dict] = []
reviews_rows: list[dict] = []

review_max_pages = (MAX_REVIEWS_PER_MOVIE // 20) + 1

with TMDBClient() as client:
    client.get_genres()
    fetch = (
        client.get_popular_movies
        if MOVIE_SOURCE == "popular"
        else client.get_top_rated_movies
    )

    for page in range(1, PAGES_END + 1):
        if len(movies_rows) >= MAX_MOVIES:
            break

        m_batch = fetch(page=page)
        for m in m_batch:
            if len(movies_rows) >= MAX_MOVIES:
                break

            movies_rows.append(movie_to_report_row(m, client))

            raw_reviews = client.get_movie_reviews(
                m.tmdb_id,
                max_pages=review_max_pages,
            )[:MAX_REVIEWS_PER_MOVIE]

            for r in raw_reviews:
                if not r.content or is_noisy_review(r.content):
                    continue
                lang = _infer_review_language(r.content)
                if ONLY_ENGLISH_REVIEWS and lang not in ("en", "unknown"):
                    continue
                reviews_rows.append(review_to_report_row(r, m.tmdb_id))

        print(
            f"Trang {page}/{PAGES_END} | phim={len(movies_rows)} | review={len(reviews_rows)}"
        )

movies_df = pd.DataFrame(movies_rows)
reviews_df = pd.DataFrame(reviews_rows)

movies_df.to_csv(OUT_MOVIES, index=False, encoding="utf-8", na_rep="")
reviews_df.to_csv(OUT_REVIEWS, index=False, encoding="utf-8", na_rep="")

print(f"Đã lưu: {OUT_MOVIES} ({len(movies_df)} dòng), {OUT_REVIEWS} ({len(reviews_df)} dòng)")
display(movies_df.head(2))
display(reviews_df.head(2))
